**How would we translate a query like this?**
``` sql
SELECT max(b) 
FROM stuff
```

**Create some fake data**


In [1]:
from faker import Faker

fake = Faker()

from dataclasses import dataclass

number_of_tuples = 40


@dataclass
class Stuff:
    a: int
    b: int


# create some fake data mimicking tuples in a relation:
data = [Stuff(fake.pyint(max_value=1000), fake.pyint(max_value=2000)) for f in range(number_of_tuples)]
data[:10]

[Stuff(a=688, b=484),
 Stuff(a=113, b=370),
 Stuff(a=833, b=464),
 Stuff(a=672, b=1653),
 Stuff(a=150, b=1519),
 Stuff(a=776, b=153),
 Stuff(a=73, b=1337),
 Stuff(a=360, b=818),
 Stuff(a=124, b=1387),
 Stuff(a=29, b=1895)]

**Algorithm 1: Full-blown Aggregation**

In [2]:
import math


def aggregate(input_data, attribute):
    """Full-blown aggregation algorithm: computes an aggregation over a list of tuples, once it is done, it returns the result"""
    current_max = -math.inf
    for el in input_data:
        current_max = max(current_max, el.__dict__[attribute])
    return current_max


aggregate(data, "b")

1947

**Algorithm 2: Online Aggregation**

In [3]:
import math


def online_aggregation(input_data, attribute):
    """Online aggregation algorithm: computes a running aggregation over a list of tuples yielding the new result after each tuple"""
    current_max = -math.inf
    for el in input_data:
        current_max = max(current_max, el.__dict__[attribute])
        yield current_max


for agg in online_aggregation(data, "b"):
    print(agg)

484
484
484
1653
1653
1653
1653
1653
1653
1895
1895
1895
1895
1895
1895
1895
1895
1895
1895
1929
1929
1929
1929
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947
1947


In [4]:
import unittest


class MyTest(unittest.TestCase):

    # the following function will be called before every test-function:

    def create_data(self, number_of_tuples):
        self.data = [Stuff(fake.pyint(max_value=20), fake.pyint(max_value=20)) for f in range(number_of_tuples)]

    def test_scans_end_to_end(self):
        for i in range(10):
            self.create_data(1000)

            res1 = aggregate(self.data, "b")
            *_, res2 = online_aggregation(self.data, "b")

            self.assertEqual(res1, res2)


# Run the unit test without shutting down the jupyter kernel
# notice that this call will look for all test classes, i.e.
# all classes inheriting from unittest.TestCase
# each of these functions will be executed once
# unittest.main(argv=['ignored', '-v'], verbosity=2, exit=False)

# only execute a specific Test-class:
unittest.main(argv=['ignored', '-v'], defaultTest='MyTest', verbosity=2, exit=False)


test_scans_end_to_end (__main__.MyTest.test_scans_end_to_end) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.021s

OK
